# Natural Language Intervention

The goal is to intervene on the NLA AV's output text or activations to see if we can force the model the reliably make mistakes on the six tasks presented in the prior experimental setup. Current ideas are

1. Lexical Corruption (do this)
-  Swap $n$ tokens for synonyms

```
Take AV output text (correct verbalization)
Pick n words, flip to antonyms
Feed to reconstructor (or whatever downstream scorer)
Measure % "incorrect" as function of n
Plot n vs. failure rate
```

- Use WordNet antonym sets
- $n$ is a fraction of eligible words, not a raw count
- must be weaker than a paraphrase
- flip from past to future

2. Activation Perturbation (dont intervene on this)
- optimize a soft or discrete perturbation on the input activation (not the prompt) to maximize the probability mass on a specific wrong verbalization. 
- The input will look like $AV(\vec{a} + \delta)$
- plot attack success rate against $||\delta||$ relative to natural activation norm at that layer, which gives a much more informative faithfulness/robustness number than a single working example
    - The strength will be modulated by $\epsilon$, which is the radius of the $L^2$ ball that $\delta$ is allowed to live in.
    - need to measure referential statistics (mean and std) of norms
    - $\epsilon$ will be a fraction of that reference
- Use PGD to find the optimal vector

```
delta = torch.zeros_like(a, requires_grad=True)
for step in range(num_steps):
    perturbed = a + delta
    loss = attack_loss_fn(AV(perturbed), correct_target)  # ascending on this
    loss.backward()
    with torch.no_grad():
        delta += lr * delta.grad.sign() * step_size   # or plain grad, not sign, if you prefer L2-style steps
        # project back into L2 ball of radius epsilon
        delta_norm = torch.norm(delta, p=2)
        if delta_norm > epsilon:
            delta = delta * (epsilon / delta_norm)
    delta.grad.zero_()
```

3. Method and Results google doc
- methodology (full text)
- results (bullet points)
- think about figure

Venues
- ICLR (16th of Sept)
- ACL (October)